# Seaborn Phase 7: Regression Diagnostics, Categorical Scatter, ECDF & Trends
### Credit Card Risk Analysis Project

This phase rounds out the Seaborn toolkit with four things worth knowing before you move
into modeling:

13. **Regplot / Lmplot / Residplot** — standalone regression plots with a fitted line
    and confidence band, plus a residual check for whether a linear fit is actually
    appropriate
14. **Stripplot / Swarmplot** — categorical scatter plots showing every individual point,
    a good complement to box/violin plots
15. **ECDFplot** — empirical cumulative distribution, the precise way to answer threshold
    questions like "what percentage of applicants have a credit score below 600?"
16. **Pointplot** — connected point estimates across ordered categories, good for
    showing a trend (like default rate) across an ordinal variable like income bracket

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid")

n = 600

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 1500, n), 500, None)
credit_score = np.clip(np.random.normal(660, 65, n), 300, 850)

total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = np.clip((total_debt / annual_income) * 100, 0, 90)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
default_probability = np.clip(raw_risk + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

risk_tier = pd.cut(credit_score, bins=[300, 600, 700, 850], labels=['High', 'Medium', 'Low'])

income_bracket = pd.cut(
    annual_income,
    bins=[0, 40000, 70000, 100000, np.inf],
    labels=['<40k', '40k-70k', '70k-100k', '100k+']
)

df = pd.DataFrame({
    'Annual_Income': annual_income,
    'Credit_Limit': credit_limit,
    'Credit_Score': credit_score,
    'Debt_to_Income': debt_to_income,
    'Default_Probability': default_probability,
    'Default': default,
    'Risk_Tier': risk_tier,
    'Income_Bracket': income_bracket
})

print(df.shape)
df.head()


**Note:** two questions in Section 13 (`logistic=True` and `lowess=True`) require the
optional `statsmodels` package. If you don't have it, install it with:
```
pip install statsmodels
```
The cell below checks whether it's available.

In [ ]:
try:
    import statsmodels
    print(f"statsmodels {statsmodels.__version__} is available — Q6 and Q8 will run as written.")
except ImportError:
    print("statsmodels is NOT installed. Run 'pip install statsmodels' before attempting "
          "Q6 and Q8, or they will raise a RuntimeError.")


---
## Section 13: Regplot, Lmplot & Residplot

You've already seen `kind='reg'` inside jointplot and pairplot. `sns.regplot()` and
`sns.lmplot()` are the standalone versions — and `sns.residplot()` answers the question
those regression lines gloss over: does a straight line actually fit this relationship
well?


**Q1.** Plot `sns.regplot(data=df, x='Annual_Income', y='Credit_Limit')` — a scatterplot with a fitted regression line and a shaded 95% confidence band, all in one call.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
sns.regplot(data=df, x='Annual_Income', y='Credit_Limit')
plt.title("Income vs. Credit Limit")
plt.show()


**Q2.** Repeat Q1, passing `ci=None` to drop the confidence band (useful when you just want the line itself, e.g. inside a busy dashboard panel) and `scatter_kws={'alpha': 0.4}` to make the underlying points semi-transparent.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
sns.regplot(data=df, x='Annual_Income', y='Credit_Limit', ci=None, scatter_kws={'alpha': 0.4})
plt.title("Income vs. Credit Limit (no CI band)")
plt.show()


**Q3.** `sns.regplot()` doesn't support `hue=` — for that you need the figure-level `sns.lmplot()`. Plot `sns.lmplot(data=df, x='Annual_Income', y='Credit_Limit', hue='Default', palette={0: 'steelblue', 1: 'tomato'})` — one regression line per Default group, both on the same axes.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
sns.lmplot(data=df, x='Annual_Income', y='Credit_Limit', hue='Default',
           palette={0: 'steelblue', 1: 'tomato'})
plt.show()


**Q4.** Repeat Q3, but use `col='Default'` instead of `hue=` — now each Default group gets its own separate panel (small multiples) rather than overlaid lines on one chart.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
sns.lmplot(data=df, x='Annual_Income', y='Credit_Limit', col='Default')
plt.show()


**Q5.** Plot `sns.regplot(data=df, x='Debt_to_Income', y='Credit_Score')` with `order=2` — fits a quadratic curve instead of a straight line, useful for checking whether a relationship you suspect is nonlinear actually bends the way you think.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
sns.regplot(data=df, x='Debt_to_Income', y='Credit_Score', order=2, scatter_kws={'alpha': 0.4})
plt.title("Debt-to-Income vs. Credit Score (quadratic fit)")
plt.show()


**Q6 (Genuinely useful for risk work).** `Default` is binary (0/1), so a straight-line fit doesn't make sense for it — but a *logistic* fit does. Plot `sns.regplot(data=df, x='Debt_to_Income', y='Default', logistic=True, scatter_kws={'alpha': 0.2})`. The resulting S-shaped curve is literally an estimate of the probability of default as a function of DTI — a direct visual preview of what a logistic regression model would learn.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
sns.regplot(data=df, x='Debt_to_Income', y='Default', logistic=True, scatter_kws={'alpha': 0.2})
plt.title("Probability of Default vs. Debt-to-Income (logistic fit)")
plt.ylabel("Default")
plt.show()


**Q7.** Plot `sns.residplot(data=df, x='Annual_Income', y='Credit_Limit')` — this fits the same linear regression as `regplot`, but plots the *residuals* (actual minus predicted) instead of the raw values. A good linear fit shows residuals scattered randomly around zero with no visible pattern.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
sns.residplot(data=df, x='Annual_Income', y='Credit_Limit')
plt.title("Residuals: Income vs. Credit Limit")
plt.axhline(0, color='black', linestyle='--')
plt.show()


**Q8.** Repeat Q7, adding `lowess=True` — this overlays a smoothed trend line through the residuals themselves. If that smoothed line is flat and near zero, the linear fit is reasonable; if it curves noticeably, that's a sign the true relationship isn't actually linear.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
sns.residplot(data=df, x='Annual_Income', y='Credit_Limit', lowess=True,
              line_kws={'color': 'red'})
plt.title("Residuals with LOWESS Trend")
plt.axhline(0, color='black', linestyle='--')
plt.show()


**Q9 (Capstone).** Build `sns.lmplot(data=df, x='Debt_to_Income', y='Credit_Score', col='Default', hue='Default', palette={0: 'seagreen', 1: 'tomato'}, height=5)` — faceted by Default *and* colored by it (redundant here, but shows both mechanisms together), then add a `g.fig.suptitle()` with `subplots_adjust` so it doesn't overlap the panels.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
g = sns.lmplot(data=df, x='Debt_to_Income', y='Credit_Score', col='Default', hue='Default',
                palette={0: 'seagreen', 1: 'tomato'}, height=5, legend=False)
g.fig.suptitle("Debt-to-Income vs. Credit Score, by Default Status")
g.fig.subplots_adjust(top=0.85)
plt.show()


> **Checkpoint — Section 13:** `regplot` is the standalone Axes-level regression plot;
> `lmplot` is its figure-level counterpart and the only one of the two that supports
> `hue=`/`col=`/`row=` faceting. `order=` catches curved relationships, `logistic=True` is
> the right tool specifically for a binary target like `Default`, and `residplot`
> (ideally with `lowess=True`) is how you check whether a linear fit is trustworthy
> instead of just assuming it.


---
## Section 14: Stripplot & Swarmplot

Box and violin plots show summary statistics, but they can hide how few — or how many —
individual points actually make up a group. Strip and swarm plots show every point
directly.


**Q10.** Plot `sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'])` — every individual applicant plotted as a point, grouped by Risk Tier, with a small random horizontal jitter so points don't all stack on one vertical line.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'])
plt.title("Debt-to-Income by Risk Tier (every applicant)")
plt.show()


**Q11.** Repeat Q10, controlling the amount of horizontal scatter directly with `jitter=0.3` (more spread, less overplotting along the vertical) versus the default.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], jitter=0.3)
plt.title("Debt-to-Income by Risk Tier (jitter=0.3)")
plt.show()


**Q12.** Plot the same data with `sns.swarmplot()` instead of `stripplot()` — swarm plots arrange points algorithmically so they never overlap at all (unlike strip plot's random jitter), at the cost of being slower and getting cramped with a lot of points.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
sns.swarmplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], size=3)
plt.title("Debt-to-Income by Risk Tier (swarm, non-overlapping)")
plt.show()


**Q13 (Common real-world pattern).** Plot `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], color='lightgray')` first, then call `sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], color='black', alpha=0.4, size=3)` on the *same* Axes — combining the summary statistics of a boxplot with the raw individual points of a strip plot in one chart.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'],
            color='lightgray', ax=ax)
sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'],
              color='black', alpha=0.4, size=3, ax=ax)
ax.set_title("Debt-to-Income by Risk Tier: Boxplot + Individual Applicants")
plt.show()


**Q14.** Plot `sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Default', order=['Low', 'Medium', 'High'], dodge=True, palette={0: 'seagreen', 1: 'tomato'})` — `dodge=True` separates the two `Default` groups side by side within each Risk Tier, instead of overlapping them in the same jittered strip.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
sns.stripplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Default',
              order=['Low', 'Medium', 'High'], dodge=True, palette={0: 'seagreen', 1: 'tomato'},
              alpha=0.6)
plt.title("Debt-to-Income by Risk Tier and Default Status")
plt.show()


**Q15.** Swarm plots get slow and visually crowded on large samples. Take a random subsample with `df.sample(150, random_state=1)` and plot a swarm plot of `Credit_Score` by `Risk_Tier` on just that subsample — a practical workaround when you want swarm plot's exact non-overlap on a dataset too large to swarm directly.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
sample_df = df.sample(150, random_state=1)
sns.swarmplot(data=sample_df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'], size=4)
plt.title("Credit Score by Risk Tier (150-applicant sample)")
plt.show()


**Q16 (Capstone).** Build the final combined chart: a `sns.violinplot()` of `Credit_Score` by `Risk_Tier` with `inner=None` (so it doesn't draw its own internal box, leaving room for the points) in a light color, then overlay a `sns.stripplot()` of the same data in black with `alpha=0.4`, `size=3`, and `jitter=0.2`. Add a bold title.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.violinplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'],
               inner=None, color='lightsteelblue', ax=ax)
sns.stripplot(data=df, x='Risk_Tier', y='Credit_Score', order=['Low', 'Medium', 'High'],
              color='black', alpha=0.4, size=3, jitter=0.2, ax=ax)
ax.set_title("Credit Score by Risk Tier: Density + Every Applicant", fontsize=13, fontweight='bold')
plt.show()


> **Checkpoint — Section 14:** Strip and swarm plots show individual observations
> rather than a summary — strip uses random jitter (fast, works at any size), swarm uses
> exact non-overlapping placement (slower, best on smaller samples). Both combine well
> as an overlay on top of a box or violin plot, giving you summary statistics and the
> raw data in one chart.


---
## Section 15: The ECDFplot

A histogram shows you *how many* observations fall in each bin. An empirical cumulative
distribution (ECDF) shows you, for any value on the x-axis, *what fraction of the data
falls at or below it* — which is exactly the shape of most real risk threshold
questions ("what fraction of applicants have DTI over 40%?").


**Q17.** Plot `sns.ecdfplot(data=df, x='Credit_Score')`. The y-axis runs from 0 to 1 — reading across from any y-value to the curve, then down to the x-axis, tells you what credit score corresponds to that cumulative proportion.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
sns.ecdfplot(data=df, x='Credit_Score')
plt.title("Credit Score: Empirical CDF")
plt.show()


**Q18.** Plot the same `Credit_Score` ECDF, then add a vertical dashed line at `x=600` with `plt.axvline(600, color='red', linestyle='--')`. Separately (not from the chart — programmatically), compute the exact proportion of applicants with `Credit_Score < 600` using `(df['Credit_Score'] < 600).mean()`, and print it to confirm what the chart shows visually.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
sns.ecdfplot(data=df, x='Credit_Score')
plt.axvline(600, color='red', linestyle='--')
plt.title("Credit Score ECDF (red line at 600)")
plt.show()

below_600 = (df['Credit_Score'] < 600).mean()
print(f"Proportion with Credit_Score < 600: {below_600:.1%}")


**Q19.** Plot `sns.ecdfplot(data=df, x='Debt_to_Income', hue='Default')` — comparing the DTI distributions of defaulted vs. paid applicants. If the two curves are clearly separated, that's strong visual evidence DTI is a useful predictor; if they're nearly on top of each other, it isn't.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
sns.ecdfplot(data=df, x='Debt_to_Income', hue='Default')
plt.title("Debt-to-Income ECDF: Defaulted vs. Paid")
plt.show()


**Q20.** Repeat Q17's `Credit_Score` ECDF, but pass `complementary=True` — this flips the curve to show the *survival function* instead: the proportion of applicants *above* each value rather than at-or-below it. Useful when the question is naturally phrased as "what fraction score above X" instead of "below X".

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
sns.ecdfplot(data=df, x='Credit_Score', complementary=True)
plt.title("Credit Score: Complementary ECDF (Survival Function)")
plt.ylabel("Proportion Above")
plt.show()


**Q21.** Create a 1x2 subplot grid: left panel a histogram of `Credit_Score` (with `kde=True`), right panel the ECDF of the same variable. Seeing them side by side makes clear that the ECDF is essentially the running cumulative sum of the histogram.

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df, x='Credit_Score', kde=True, ax=axes[0])
axes[0].set_title("Histogram + KDE")

sns.ecdfplot(data=df, x='Credit_Score', ax=axes[1])
axes[1].set_title("ECDF")

plt.show()


**Q22.** Repeat Q17's ECDF, but pass `stat='count'` instead of the default `'proportion'` — now the y-axis shows the raw cumulative count of applicants rather than a 0-to-1 proportion.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
sns.ecdfplot(data=df, x='Credit_Score', stat='count')
plt.title("Credit Score: Cumulative Count")
plt.show()


**Q23 (Capstone).** Build a chart that directly answers a specific risk threshold question: plot `sns.ecdfplot(data=df, x='Debt_to_Income')`, add a vertical line at `x=40` and a horizontal line at the corresponding proportion, and annotate the intersection with `plt.annotate()` stating the exact percentage of applicants with DTI over 40% (computed programmatically, not read off the chart).

In [ ]:
# YOUR CODE HERE


**Solution 23**

In [ ]:
pct_above_40 = (df['Debt_to_Income'] > 40).mean() * 100
pct_at_or_below_40 = 100 - pct_above_40

fig, ax = plt.subplots(figsize=(9, 6))
sns.ecdfplot(data=df, x='Debt_to_Income', ax=ax)
ax.axvline(40, color='red', linestyle='--')
ax.axhline(pct_at_or_below_40 / 100, color='red', linestyle='--')
ax.annotate(
    f"{pct_above_40:.1f}% of applicants\nhave DTI > 40%",
    xy=(40, pct_at_or_below_40 / 100),
    xytext=(48, pct_at_or_below_40 / 100 - 0.15),
    arrowprops=dict(facecolor='black', arrowstyle='->')
)
ax.set_title("Debt-to-Income: Empirical CDF", fontsize=13, fontweight='bold')
plt.show()


> **Checkpoint — Section 15:** `sns.ecdfplot()` directly answers "what fraction of my
> data is above/below this threshold," which is precisely how most risk cutoffs get
> discussed. It's more precise than reading a histogram, needs no bin choice at all, and
> `hue=` makes group comparison just as easy as with a histogram.


---
## Section 16: The Pointplot

`sns.pointplot()` computes an estimator per category, same as `barplot` — but instead of
separate bars, it draws connected points. That connecting line is the whole point: it
makes a *trend* across ordered categories immediately visible in a way bars don't.


**Q24.** Plot `sns.pointplot(data=df, x='Income_Bracket', y='Default', order=['<40k', '40k-70k', '70k-100k', '100k+'])`. Since `Default` is 0/1, the plotted value at each point is the *default rate* for that income bracket — and the connecting line shows whether that rate rises, falls, or has no clear trend across brackets.

In [ ]:
# YOUR CODE HERE


**Solution 24**

In [ ]:
bracket_order = ['<40k', '40k-70k', '70k-100k', '100k+']

sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order)
plt.title("Default Rate by Income Bracket")
plt.ylabel("Default Rate")
plt.show()


**Q25.** Repeat Q24, adding `hue='Risk_Tier'` — now you get one connected line per Risk Tier, letting you see whether the income-bracket trend in default rate holds consistently across all three risk tiers or differs between them.

In [ ]:
# YOUR CODE HERE


**Solution 25**

In [ ]:
sns.pointplot(data=df, x='Income_Bracket', y='Default', hue='Risk_Tier',
              order=bracket_order, hue_order=['Low', 'Medium', 'High'])
plt.title("Default Rate by Income Bracket and Risk Tier")
plt.show()


**Q26.** Repeat Q24, customizing the appearance with `markers='D'` (diamond markers instead of circles), `linestyles='--'` (dashed connecting line), and `color='darkred'`.

In [ ]:
# YOUR CODE HERE


**Solution 26**

In [ ]:
sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order,
              markers='D', linestyles='--', color='darkred')
plt.title("Default Rate by Income Bracket")
plt.show()


**Q27.** Repeat Q24, but pass `errorbar=None` to drop the confidence interval whiskers entirely — a cleaner look once you've already established the uncertainty elsewhere and just want the trend line itself to stand out.

In [ ]:
# YOUR CODE HERE


**Solution 27**

In [ ]:
sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order, errorbar=None)
plt.title("Default Rate by Income Bracket (no error bars)")
plt.show()


**Q28.** Create a 1x2 subplot grid comparing `sns.barplot()` and `sns.pointplot()` on the exact same data (`Default` rate by `Income_Bracket`), side by side. Which one makes the trend across brackets easier to read at a glance?

In [ ]:
# YOUR CODE HERE


**Solution 28**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

sns.barplot(data=df, x='Income_Bracket', y='Default', order=bracket_order, ax=axes[0])
axes[0].set_title("Barplot")

sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order, ax=axes[1])
axes[1].set_title("Pointplot")

fig.suptitle("Default Rate by Income Bracket: Two Views")
plt.show()

# The pointplot's connecting line makes the downward trend across brackets easier to
# read at a glance than comparing separate bar heights.


**Q29 (Capstone).** Build the final version: `sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order, color='darkred', markers='o')`, a bold title, a y-label of `"Default Rate"`, and `sns.despine()`. Finish by printing the exact default rate per bracket with `df.groupby('Income_Bracket', observed=True)['Default'].mean()` so the chart's trend is backed by exact numbers.

In [ ]:
# YOUR CODE HERE


**Solution 29**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.pointplot(data=df, x='Income_Bracket', y='Default', order=bracket_order,
              color='darkred', markers='o', ax=ax)
ax.set_title("Default Rate by Income Bracket", fontsize=13, fontweight='bold')
ax.set_ylabel("Default Rate")
sns.despine(ax=ax)
plt.show()

print(df.groupby('Income_Bracket', observed=True)['Default'].mean())


---
## Checkpoint: Phase 7 Complete

You've covered:
- **Regplot / Lmplot / Residplot** — standalone regression visuals, `logistic=True` for
  a binary target like `Default`, and residual checks (ideally with `lowess=True`)
  before trusting a linear relationship
- **Stripplot / Swarmplot** — every individual observation, either jittered (fast, any
  size) or exactly non-overlapping (slower, best on smaller samples), and both work well
  layered on top of a box or violin plot
- **ECDFplot** — the precise way to answer "what fraction of applicants are above/below
  this threshold," with no bin-width choice needed
- **Pointplot** — connected estimates across ordered categories, making a trend (like
  default rate rising or falling across income brackets) visually obvious in a way bars
  alone don't

Between everything covered in Phases 1-7, you now have the complete Matplotlib +
Seaborn toolkit this project needs — from a single line plot all the way through
regression diagnostics and multivariate exploration.

**Where to next:** this is a genuinely strong stopping point for visualization. From
here, the natural move is applying this full toolkit directly to your real credit card
risk dataset, or moving into feature engineering and modeling.
